# 07 - Dados Reais de Produção Volve (Noruega)

Notebook híbrido para carregar dados reais do arquivo `volve_ml_ready.csv`, persisti-los em `SQLite`, gerar SQL com um modelo local no `Ollama` e usar `OpenRouter` apenas para redigir a resposta final.

A base representa uma série temporal real de produção de um poço offshore do projeto Volve, no Mar do Norte da Noruega.

## Objetivo
- Ler o arquivo `volve_ml_ready.csv` em um `pandas DataFrame`.
- Exibir colunas, `shape`, `describe()` e `head(10)`.
- Salvar os dados em um banco `SQLite` com o mesmo nome-base do arquivo CSV.
- Traduzir perguntas em linguagem natural para SQL usando `Qwen2.5-Coder` local.
- Executar consultas em `SQLite` localmente sobre os dados reais.
- Usar `Claude Sonnet 4.6` no OpenRouter apenas para a resposta final.

## Ordem de execução

1. Garanta que o `Ollama` esteja instalado e com o serviço iniciado: `ollama serve`.
2. Baixe o modelo local para geração de SQL: `ollama pull qwen2.5-coder:7b-instruct`.
3. Defina a variável de ambiente `OPENROUTER_API_KEY`.
4. Execute a célula de carga do CSV, inspeção do `DataFrame`, gravação em `SQLite` e montagem do dicionário de dados.
5. Execute a célula de configuração dos modelos e do estado do agente.
6. Execute a célula com os nós do fluxo e compile o grafo.
7. Execute a célula final para rodar o teste de estresse e gerar o relatório.

Observação: se a sua CPU ficar muito lenta com `qwen2.5-coder:7b-instruct`, troque `LOCAL_SQL_MODEL` para `qwen2.5-coder:3b-instruct`.

## Fonte dos Dados

O arquivo `volve_ml_ready.csv` é tratado neste notebook como uma versão preparada para analytics e machine learning derivada do **Volve Field Dataset**.

### Proveniência reconhecida

- Base pública associada ao projeto **Volve Field Dataset**.
- Dados liberados pela **Equinor** (antiga **Statoil**).
- Campo **Volve**, offshore da Noruega, no **Mar do Norte**, bloco **15/9**.
- Janela histórica de produção do campo: **2008 a 2016**.
- Unidade de produção associada ao campo: **Mærsk Inspirer**.

### Assinatura típica da base Volve

Essa família de dados é normalmente reconhecida por colunas como:

- `DATEPRD`
- `WELL_BORE_CODE`
- `NPD_WELL_BORE_CODE`
- `NPD_WELL_BORE_NAME`
- `NPD_FIELD_NAME`
- `NPD_FACILITY_NAME`
- `ON_STREAM_HRS`
- `AVG_DOWNHOLE_PRESSURE`
- `AVG_DP_TUBING`
- `AVG_CHOKE_SIZE_P`
- `AVG_WHP_P`
- `AVG_WHT_P`
- `AVG_WELL_BORE_OIL_VOL`
- `AVG_WELL_BORE_GAS_VOL`
- `AVG_WELL_BORE_WAT_VOL`

O arquivo local usado aqui parece ser uma **versão transformada e reduzida** dessa base, já adaptada para uso analítico e para tarefas de machine learning. Por isso, parte das colunas regulatórias e operacionais clássicas pode não aparecer mais no CSV final, enquanto outras colunas derivadas foram adicionadas.

### Fontes históricas associadas a essa família de dados

- Portal oficial **Volve Data Village**.
- Base regulatória da antiga **NPD** (*Norwegian Petroleum Directorate*), hoje **NOD** (*Norwegian Offshore Directorate*).
- Espelhos e reempacotamentos públicos em **GitHub** usados por pesquisadores.

### Relevância técnica

O dataset Volve se tornou uma base de referência em Oil & Gas para estudos de:

- *Production Surveillance*
- *Well Performance Analytics*
- *Forecasting*
- *Decline Curve Analysis*
- *Anomaly Detection*
- *Machine Learning para Produção*
- *Digital Oilfield*


### Célula 1: Carga do CSV, inspeção do DataFrame e preparação do banco

In [1]:
import os
import sqlite3
import time
from typing import Any, Dict, List

import ollama
import pandas as pd
from IPython.display import display
from openai import OpenAI
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict


NOTEBOOKS_DIR = os.path.join(os.getcwd(), "notebooks")
DATA_DIR = NOTEBOOKS_DIR if os.path.isdir(NOTEBOOKS_DIR) else os.getcwd()
CSV_FILE_NAME = "volve_ml_ready.csv"
CSV_PATH = os.path.join(DATA_DIR, CSV_FILE_NAME)
TABLE_NAME = os.path.splitext(CSV_FILE_NAME)[0]
DB_NAME = f"{TABLE_NAME}.db"
DB_PATH = os.path.join(DATA_DIR, DB_NAME)

if "conn" in globals():
    try:
        conn.close()
    except Exception:
        pass

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Arquivo CSV não encontrado: {CSV_PATH}")

if os.path.exists(DB_PATH):
    try:
        os.remove(DB_PATH)
    except OSError as exc:
        print(f"[Aviso] Banco antigo não pôde ser removido: {exc}")

source_df = pd.read_csv(CSV_PATH)

print(f"[DADOS] CSV carregado de: {CSV_PATH}")
print(f"[DADOS] Banco SQLite será salvo em: {DB_PATH}")
print("\n[INSPEÇÃO] Colunas do DataFrame:")
print(source_df.columns.tolist())
print(f"\n[INSPEÇÃO] Shape: {source_df.shape}")
print("\n[INSPEÇÃO] Describe:")
display(source_df.describe(include="all"))
print("\n[INSPEÇÃO] Head(10):")
display(source_df.head(10))

setup_conn = sqlite3.connect(DB_PATH)
source_df.to_sql(TABLE_NAME, setup_conn, if_exists="replace", index=False)

schema_df = pd.read_sql(f"PRAGMA table_info({TABLE_NAME})", setup_conn)
schema_text = ", ".join(
    f"{row['name']} ({row['type'] or 'TEXT'})"
    for _, row in schema_df.iterrows()
)

BASE_COLUMN_METADATA = {
    "DATEPRD": {
        "descricao": "Data da producao/operacao diaria.",
        "tipo_analitico": "datetime",
        "unidade": "data",
        "origem": "Sistema operacional / historian",
        "local_medicao": "Centro de supervisao / banco operacional",
        "classe": "coluna original do dataset Volve",
    },
    "WELL_TYPE": {
        "descricao": "Tipo do poco, por exemplo produtor ou injetor.",
        "tipo_analitico": "string",
        "unidade": "N/A",
        "origem": "Engenharia de producao",
        "local_medicao": "Configuracao operacional do poco",
        "classe": "coluna original do dataset Volve",
    },
    "ON_STREAM_HRS": {
        "descricao": "Quantidade de horas em operacao/produzindo no dia.",
        "tipo_analitico": "float",
        "unidade": "horas",
        "origem": "Sistema supervisorio / producao",
        "local_medicao": "Status operacional do poco",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_DOWNHOLE_PRESSURE": {
        "descricao": "Pressao media no fundo do poco.",
        "tipo_analitico": "float",
        "unidade": "bar(a)",
        "origem": "Gauge de fundo / sensor downhole",
        "local_medicao": "Fundo do poco / proximo da zona produtora",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_DOWNHOLE_TEMPERATURE": {
        "descricao": "Temperatura media no fundo do poco.",
        "tipo_analitico": "float",
        "unidade": "graus C",
        "origem": "Sensor downhole",
        "local_medicao": "Fundo do poco / tubing inferior",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_DP_TUBING": {
        "descricao": "Delta de pressao medio no tubing.",
        "tipo_analitico": "float",
        "unidade": "bar",
        "origem": "Sensores de pressao no tubing",
        "local_medicao": "Interior do tubing de producao",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_CHOKE_SIZE_P": {
        "descricao": "Abertura media do choke em superficie.",
        "tipo_analitico": "float",
        "unidade": "%",
        "origem": "Atuador/sensor do choke",
        "local_medicao": "Choke na arvore de natal / superficie",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_WHP_P": {
        "descricao": "Pressao media na cabeca do poco.",
        "tipo_analitico": "float",
        "unidade": "bar",
        "origem": "Sensor wellhead",
        "local_medicao": "Cabeca do poco / arvore de natal",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_WHT_P": {
        "descricao": "Temperatura media na cabeca do poco.",
        "tipo_analitico": "float",
        "unidade": "graus C",
        "origem": "Sensor wellhead",
        "local_medicao": "Cabeca do poco / arvore de natal",
        "classe": "coluna original do dataset Volve",
    },
    "BORE_OIL_VOL": {
        "descricao": "Volume diario de oleo produzido.",
        "tipo_analitico": "float",
        "unidade": "Sm3/d",
        "origem": "Medidor multifasico / teste de producao",
        "local_medicao": "Linha de producao do poco",
        "classe": "coluna original do dataset Volve",
    },
    "BORE_WAT_VOL": {
        "descricao": "Volume diario de agua produzida.",
        "tipo_analitico": "float",
        "unidade": "Sm3/d",
        "origem": "Medidor multifasico / separador",
        "local_medicao": "Linha de producao / separador",
        "classe": "coluna original do dataset Volve",
    },
}

FEATURE_BASE_INFO = {
    "oil": {
        "descricao_base": "serie de oleo derivada de BORE_OIL_VOL",
        "unidade": "Sm3/d",
        "origem": "Feature engineering a partir de BORE_OIL_VOL",
    },
    "water": {
        "descricao_base": "serie de agua derivada de BORE_WAT_VOL",
        "unidade": "Sm3/d",
        "origem": "Feature engineering a partir de BORE_WAT_VOL",
    },
    "gas": {
        "descricao_base": "serie de gas derivada da familia BORE_GAS_VOL do dataset Volve original",
        "unidade": "Sm3/d",
        "origem": "Feature engineering a partir da serie de gas do dataset fonte",
    },
}

def infer_feature_metadata(column_name: str) -> Dict[str, str]:
    lower_name = column_name.lower()

    for prefix, base_info in FEATURE_BASE_INFO.items():
        if lower_name.startswith(f"{prefix}_lag_"):
            days = lower_name.split("_")[-1]
            return {
                "descricao": f"Valor defasado em {days} dias da {base_info['descricao_base']}.",
                "tipo_analitico": "feature derivada temporal",
                "unidade": base_info["unidade"],
                "origem": base_info["origem"],
                "local_medicao": "Derivada em pipeline analitico / data science",
                "classe": "feature derivada - lag temporal",
            }
        if lower_name.startswith(f"{prefix}_roll_mean_"):
            window = lower_name.split("_")[-1]
            return {
                "descricao": f"Media movel de {window} dias da {base_info['descricao_base']}.",
                "tipo_analitico": "feature derivada rolling mean",
                "unidade": base_info["unidade"],
                "origem": base_info["origem"],
                "local_medicao": "Derivada em pipeline analitico / data science",
                "classe": "feature derivada - rolling mean",
            }
        if lower_name.startswith(f"{prefix}_roll_std_"):
            window = lower_name.split("_")[-1]
            return {
                "descricao": f"Desvio padrao movel de {window} dias da {base_info['descricao_base']}.",
                "tipo_analitico": "feature derivada rolling std",
                "unidade": base_info["unidade"],
                "origem": base_info["origem"],
                "local_medicao": "Derivada em pipeline analitico / data science",
                "classe": "feature derivada - rolling std",
            }
        if lower_name.startswith(f"{prefix}_delta_"):
            days = lower_name.split("_")[-1].replace("d", "")
            return {
                "descricao": f"Diferenca da {base_info['descricao_base']} em relacao a {days} dias antes.",
                "tipo_analitico": "feature derivada delta",
                "unidade": base_info["unidade"],
                "origem": base_info["origem"],
                "local_medicao": "Derivada em pipeline analitico / data science",
                "classe": "feature derivada - delta temporal",
            }
        if lower_name.startswith(f"{prefix}_pct_change_"):
            days = lower_name.split("_")[-1].replace("d", "")
            return {
                "descricao": f"Variacao percentual da {base_info['descricao_base']} em relacao a {days} dias antes.",
                "tipo_analitico": "feature derivada percentual",
                "unidade": "%",
                "origem": base_info["origem"],
                "local_medicao": "Derivada em pipeline analitico / data science",
                "classe": "feature derivada - pct change",
            }

    if lower_name == "oil_roll_30":
        return {
            "descricao": "Feature rolling de 30 dias aplicada a serie de oleo.",
            "tipo_analitico": "feature derivada rolling",
            "unidade": "Sm3/d",
            "origem": "Feature engineering a partir de BORE_OIL_VOL",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - rolling",
        }
    if lower_name == "oil_expanding_mean":
        return {
            "descricao": "Media expansiva acumulada da serie de oleo ao longo do tempo.",
            "tipo_analitico": "feature derivada expanding mean",
            "unidade": "Sm3/d",
            "origem": "Feature engineering a partir de BORE_OIL_VOL",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - expanding mean",
        }
    if lower_name == "oil_expanding_std":
        return {
            "descricao": "Desvio padrao expansivo acumulado da serie de oleo ao longo do tempo.",
            "tipo_analitico": "feature derivada expanding std",
            "unidade": "Sm3/d",
            "origem": "Feature engineering a partir de BORE_OIL_VOL",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - expanding std",
        }
    if lower_name == "water_cumulative":
        return {
            "descricao": "Acumulado historico da serie de agua produzida ao longo do periodo.",
            "tipo_analitico": "feature derivada cumulativa",
            "unidade": "Sm3 acumulado",
            "origem": "Feature engineering a partir de BORE_WAT_VOL",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - cumulativa",
        }
    if lower_name == "oil_acceleration":
        return {
            "descricao": "Indicador derivado de aceleracao da dinamica da serie de oleo.",
            "tipo_analitico": "feature derivada de segunda ordem",
            "unidade": "unidade derivada do pipeline analitico",
            "origem": "Feature engineering a partir de BORE_OIL_VOL",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - aceleracao",
        }
    if lower_name in {"oil_trend_strength", "water_trend_strength"}:
        family = "oleo" if lower_name.startswith("oil") else "agua"
        return {
            "descricao": f"Indicador derivado de forca de tendencia da serie de {family}.",
            "tipo_analitico": "feature derivada de tendencia",
            "unidade": "unidade derivada do pipeline analitico",
            "origem": "Feature engineering temporal",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - trend strength",
        }
    if lower_name in {"oil_vs_trend", "water_vs_trend"}:
        family = "oleo" if lower_name.startswith("oil") else "agua"
        return {
            "descricao": f"Razao entre o valor corrente e a tendencia estimada da serie de {family}.",
            "tipo_analitico": "feature derivada de comparacao com tendencia",
            "unidade": "adimensional",
            "origem": "Feature engineering temporal",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - vs trend",
        }
    if lower_name == "oil_volatility_index":
        return {
            "descricao": "Indice derivado de volatilidade da serie de oleo.",
            "tipo_analitico": "feature derivada de volatilidade",
            "unidade": "adimensional",
            "origem": "Feature engineering temporal",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - volatility index",
        }
    if lower_name == "oil_momentum_30d":
        return {
            "descricao": "Momentum de 30 dias da serie de oleo, comparando o valor corrente com a referencia de 30 dias antes.",
            "tipo_analitico": "feature derivada de momentum",
            "unidade": "Sm3/d",
            "origem": "Feature engineering a partir de BORE_OIL_VOL",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - momentum",
        }
    if lower_name == "oil_roc_30d":
        return {
            "descricao": "Rate of change de 30 dias da serie de oleo.",
            "tipo_analitico": "feature derivada de taxa de variacao",
            "unidade": "fracao ou % conforme convencao do pipeline",
            "origem": "Feature engineering a partir de BORE_OIL_VOL",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - rate of change",
        }
    if lower_name == "oil_zscore_30":
        return {
            "descricao": "Z-score de 30 dias da serie de oleo.",
            "tipo_analitico": "feature derivada de padronizacao",
            "unidade": "adimensional",
            "origem": "Feature engineering a partir de BORE_OIL_VOL",
            "local_medicao": "Derivada em pipeline analitico / data science",
            "classe": "feature derivada - zscore",
        }

    return {
        "descricao": "Coluna derivada presente no CSV tratado para analytics e machine learning.",
        "tipo_analitico": "feature derivada",
        "unidade": "dependente da transformacao",
        "origem": "Pipeline analitico / feature engineering",
        "local_medicao": "Derivada em ambiente analitico",
        "classe": "feature derivada",
    }

def get_column_metadata(column_name: str, column_type: str) -> Dict[str, str]:
    metadata = BASE_COLUMN_METADATA.get(column_name)
    if metadata is not None:
        return metadata
    return infer_feature_metadata(column_name)

data_dictionary_lines = [
    f"TABELA: {TABLE_NAME}",
    "CONTEXTO: Serie temporal real de producao de um poco offshore do projeto Volve, no Mar do Norte da Noruega, carregada a partir de arquivo CSV e persistida em SQLite.",
    "",
    "OBSERVACOES GERAIS:",
    "- Esta tabela contem apenas as colunas disponiveis no CSV tratado volve_ml_ready.csv.",
    "- Algumas colunas classicas do dataset bruto Volve nao estao presentes nesta versao tratada.",
    "- Colunas prefixadas com oil_, water_ e gas_ sao features derivadas usadas para analytics e machine learning.",
    "- Para SQL, use apenas os nomes de coluna listados abaixo exatamente como aparecem.",
    "",
    "COLUNAS DISPONIVEIS NESTA TABELA:",
]
for _, row in schema_df.iterrows():
    column_name = str(row["name"])
    column_type = str(row["type"] or "TEXT")
    metadata = get_column_metadata(column_name, column_type)
    line = " | ".join(
        [
            f"- {column_name}",
            f"sql_type={column_type}",
            f"classe={metadata['classe']}",
            f"descricao={metadata['descricao']}",
            f"tipo_analitico={metadata['tipo_analitico']}",
            f"unidade={metadata['unidade']}",
            f"origem={metadata['origem']}",
            f"local_medicao={metadata['local_medicao']}",
        ]
    )
    data_dictionary_lines.append(line)

DATA_DICTIONARY = "\n".join(data_dictionary_lines)

def build_query_column_context(columns: List[str]) -> str:
    lines = ["COLUNAS RETORNADAS PELA CONSULTA:"]
    for column_name in columns:
        metadata = get_column_metadata(column_name, "RESULT")
        line = " | ".join(
            [
                f"- {column_name}",
                f"classe={metadata['classe']}",
                f"descricao={metadata['descricao']}",
                f"tipo_analitico={metadata['tipo_analitico']}",
                f"unidade={metadata['unidade']}",
            ]
        )
        lines.append(line)
    return "\n".join(lines)
DATA_SOURCE_TEXT = """
BASE: volve_ml_ready.csv
PROVENIÊNCIA: versão preparada para analytics e machine learning derivada do Volve Field Dataset.
LIBERAÇÃO PÚBLICA ORIGINAL: Equinor (antiga Statoil).
CONTEXTO OPERACIONAL:
- Campo Volve, offshore da Noruega, no Mar do Norte, bloco 15/9.
- Produção histórica do campo entre 2008 e 2016.
- Unidade de produção associada: Mærsk Inspirer.
OBSERVAÇÕES:
- O CSV local usado neste notebook é uma versão tratada para analytics e ML.
- Parte das colunas clássicas do dataset Volve pode ter sido transformada, removida ou enriquecida durante a preparação.
- A família de dados Volve ficou conhecida por disponibilizar séries reais de produção e variáveis operacionais para pesquisa e indústria.
FONTES HISTÓRICAS ASSOCIADAS À FAMÍLIA DE DADOS:
- Volve Data Village.
- Base regulatória NPD, atualmente NOD.
- Espelhos públicos em GitHub usados por pesquisadores.
""".strip()
setup_conn.close()

conn = sqlite3.connect(
    f"file:{DB_PATH}?mode=ro",
    uri=True,
    check_same_thread=False,
)

print("\n[GOVERNANÇA] Banco de dados inicializado em modo read-only.")
display(schema_df)
db_preview_df = pd.read_sql(f'SELECT * FROM "{TABLE_NAME}" LIMIT 10', conn)
print("\n[VALIDAÇÃO] Leitura dos 10 primeiros registros a partir do SQLite:")
display(db_preview_df)

[DADOS] CSV carregado de: /home/wolf/Documentos/lab-artificial-inteligence/notebooks/volve_ml_ready.csv
[DADOS] Banco SQLite será salvo em: /home/wolf/Documentos/lab-artificial-inteligence/notebooks/volve_ml_ready.db

[INSPEÇÃO] Colunas do DataFrame:
['DATEPRD', 'ON_STREAM_HRS', 'AVG_DOWNHOLE_PRESSURE', 'AVG_DOWNHOLE_TEMPERATURE', 'AVG_DP_TUBING', 'AVG_CHOKE_SIZE_P', 'AVG_WHP_P', 'AVG_WHT_P', 'BORE_OIL_VOL', 'BORE_WAT_VOL', 'WELL_TYPE', 'oil_roll_30', 'oil_lag_1', 'water_lag_1', 'oil_lag_3', 'water_lag_3', 'oil_lag_7', 'water_lag_7', 'oil_lag_14', 'water_lag_14', 'oil_lag_30', 'gas_lag_30', 'water_lag_30', 'oil_roll_mean_3', 'oil_roll_mean_7', 'water_roll_mean_7', 'oil_roll_mean_14', 'water_roll_mean_30', 'oil_roll_std_7', 'water_roll_std_7', 'oil_roll_std_14', 'water_roll_std_14', 'oil_roll_std_30', 'water_roll_std_30', 'oil_delta_1d', 'water_delta_1d', 'oil_delta_3d', 'water_delta_3d', 'oil_delta_7d', 'water_delta_7d', 'oil_pct_change_1d', 'water_pct_change_1d', 'oil_pct_change_7d', 

,DATEPRD,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_OIL_VOL,BORE_WAT_VOL,...,water_cumulative,oil_acceleration,oil_trend_strength,water_trend_strength,oil_vs_trend,water_vs_trend,oil_volatility_index,oil_momentum_30d,oil_roc_30d,oil_zscore_30
count,125,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.00000,...,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.00000,125.000000,125.000000
unique,125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2014-06-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,23.648800,217.614074,108.207550,172.940248,48.577485,44.673826,56.494484,428.729760,235.05424,...,17311.774800,-5.497360,-28.274622,24.682399,0.959332,1.226105,0.138246,-85.93680,-0.071723,-0.653451
std,NaN,2.307786,3.404902,0.157091,11.141340,4.271622,9.224183,4.048699,161.042726,125.96291,...,16672.230599,117.268766,60.725764,51.040085,0.337634,0.494197,0.129095,143.82898,0.597412,1.163929
min,NaN,0.991660,209.822862,107.447354,154.685158,15.614031,28.380119,44.267469,0.000000,0.00000,...,1252.950000,-448.650000,-158.939190,-85.579762,0.000000,0.000000,0.008125,-705.74000,-1.000000,-5.113725
25%,NaN,24.000000,215.154616,108.134254,162.789418,47.607174,36.665793,54.294324,296.200000,94.65000,...,3482.520000,-13.500000,-74.273381,-6.772524,0.840121,0.969086,0.045412,-169.01000,-0.279748,-1.155068
50%,NaN,24.000000,216.044918,108.202384,171.445014,48.698561,42.617301,56.696216,378.380000,247.16000,...,10782.010000,-1.680000,-21.787857,18.386571,0.939541,1.071721,0.107189,-52.27000,-0.119289,-0.594634
75%,NaN,24.000000,220.275472,108.342534,183.488217,50.800792,52.663308,58.737828,518.320000,323.66000,...,29999.660000,7.550000,7.547429,35.614238,1.061093,1.337841,0.201331,-23.09000,-0.044570,0.253275



[INSPEÇÃO] Head(10):


,DATEPRD,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_OIL_VOL,BORE_WAT_VOL,...,water_cumulative,oil_acceleration,oil_trend_strength,water_trend_strength,oil_vs_trend,water_vs_trend,oil_volatility_index,oil_momentum_30d,oil_roc_30d,oil_zscore_30
0,2014-06-06,24.0,220.749795,107.986806,157.674225,47.202699,63.075569,56.534234,690.38,92.18,...,1252.95,42.60,-53.398714,36.586429,0.954443,2.207111,0.101595,-234.93,-0.253893,-0.251521
1,2014-06-07,24.0,220.640141,107.994109,157.278305,47.194107,63.361836,56.615421,694.30,92.07,...,1345.02,-9.17,-49.170048,35.386000,0.969673,2.053575,0.099854,-219.56,-0.240256,-0.172277
2,2014-06-08,24.0,220.504050,108.002713,157.184594,47.152024,63.319457,52.051159,690.70,91.25,...,1436.27,-7.52,-71.087190,33.940048,0.942106,1.905979,0.097347,513.90,2.906674,-0.568233
3,2014-06-09,19.5,221.450514,107.757260,159.113239,38.978911,62.337275,53.559224,238.45,28.15,...,1464.42,-448.65,-79.188238,30.421714,0.336025,0.576679,0.208615,-705.74,-0.747455,-4.317439
4,2014-06-10,24.0,220.851692,107.990754,158.074501,46.512532,62.777191,54.486153,647.11,84.88,...,1549.30,860.91,-83.024429,28.189524,0.919526,1.643581,0.208276,-176.30,-0.214110,-0.526682
5,2014-06-11,24.0,218.414712,108.065274,156.685164,47.815208,61.729549,54.295347,745.92,94.23,...,1643.53,-309.85,-76.666190,27.119952,1.061093,1.720017,0.210288,-23.09,-0.030026,0.400897
6,2014-06-12,24.0,215.496859,108.139361,155.559215,48.727774,59.937644,61.561027,804.85,101.88,...,1745.41,-39.88,-60.578333,25.339667,1.141456,1.751107,0.216511,64.05,0.086461,0.918978
7,2014-06-13,24.0,214.823228,108.135199,154.685158,48.263917,60.138070,62.327179,818.89,56.75,...,1802.16,-44.89,-45.741429,18.386571,1.155596,0.944700,0.222879,105.65,0.148127,0.997787
8,2014-06-14,24.0,215.832541,108.104783,155.597785,48.005914,60.234755,59.584407,781.21,54.32,...,1856.48,-51.72,-34.571714,11.183048,1.100488,0.877790,0.225316,37.38,0.050253,0.641931
9,2014-06-15,24.0,214.877321,108.113804,154.957708,48.040325,59.919612,60.774161,786.16,54.55,...,1911.03,42.63,-22.603571,4.121857,1.104863,0.856344,0.227629,50.07,0.068022,0.666773



[GOVERNANÇA] Banco de dados inicializado em modo read-only.


,cid,name,type,notnull,dflt_value,pk
0,0,DATEPRD,TEXT,0,None,0
1,1,ON_STREAM_HRS,REAL,0,None,0
2,2,AVG_DOWNHOLE_PRESSURE,REAL,0,None,0
3,3,AVG_DOWNHOLE_TEMPERATURE,REAL,0,None,0
4,4,AVG_DP_TUBING,REAL,0,None,0
5,5,AVG_CHOKE_SIZE_P,REAL,0,None,0
6,6,AVG_WHP_P,REAL,0,None,0
7,7,AVG_WHT_P,REAL,0,None,0
8,8,BORE_OIL_VOL,REAL,0,None,0
9,9,BORE_WAT_VOL,REAL,0,None,0



[VALIDAÇÃO] Leitura dos 10 primeiros registros a partir do SQLite:


,DATEPRD,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_OIL_VOL,BORE_WAT_VOL,...,water_cumulative,oil_acceleration,oil_trend_strength,water_trend_strength,oil_vs_trend,water_vs_trend,oil_volatility_index,oil_momentum_30d,oil_roc_30d,oil_zscore_30
0,2014-06-06,24.0,220.749795,107.986806,157.674225,47.202699,63.075569,56.534234,690.38,92.18,...,1252.95,42.60,-53.398714,36.586429,0.954443,2.207111,0.101595,-234.93,-0.253893,-0.251521
1,2014-06-07,24.0,220.640141,107.994109,157.278305,47.194107,63.361836,56.615421,694.30,92.07,...,1345.02,-9.17,-49.170048,35.386000,0.969673,2.053575,0.099854,-219.56,-0.240256,-0.172277
2,2014-06-08,24.0,220.504050,108.002713,157.184594,47.152024,63.319457,52.051159,690.70,91.25,...,1436.27,-7.52,-71.087190,33.940048,0.942106,1.905979,0.097347,513.90,2.906674,-0.568233
3,2014-06-09,19.5,221.450514,107.757260,159.113239,38.978911,62.337275,53.559224,238.45,28.15,...,1464.42,-448.65,-79.188238,30.421714,0.336025,0.576679,0.208615,-705.74,-0.747455,-4.317439
4,2014-06-10,24.0,220.851692,107.990754,158.074501,46.512532,62.777191,54.486153,647.11,84.88,...,1549.30,860.91,-83.024429,28.189524,0.919526,1.643581,0.208276,-176.30,-0.214110,-0.526682
5,2014-06-11,24.0,218.414712,108.065274,156.685164,47.815208,61.729549,54.295347,745.92,94.23,...,1643.53,-309.85,-76.666190,27.119952,1.061093,1.720017,0.210288,-23.09,-0.030026,0.400897
6,2014-06-12,24.0,215.496859,108.139361,155.559215,48.727774,59.937644,61.561027,804.85,101.88,...,1745.41,-39.88,-60.578333,25.339667,1.141456,1.751107,0.216511,64.05,0.086461,0.918978
7,2014-06-13,24.0,214.823228,108.135199,154.685158,48.263917,60.138070,62.327179,818.89,56.75,...,1802.16,-44.89,-45.741429,18.386571,1.155596,0.944700,0.222879,105.65,0.148127,0.997787
8,2014-06-14,24.0,215.832541,108.104783,155.597785,48.005914,60.234755,59.584407,781.21,54.32,...,1856.48,-51.72,-34.571714,11.183048,1.100488,0.877790,0.225316,37.38,0.050253,0.641931
9,2014-06-15,24.0,214.877321,108.113804,154.957708,48.040325,59.919612,60.774161,786.16,54.55,...,1911.03,42.63,-22.603571,4.121857,1.104863,0.856344,0.227629,50.07,0.068022,0.666773


### Célula 2: Configuração do modelo e estado do agente

In [2]:
LOCAL_SQL_MODEL = os.environ.get("LOCAL_SQL_MODEL", "qwen2.5-coder:7b-instruct")
REMOTE_RESPONSE_MODEL = "anthropic/claude-sonnet-4.6"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_TIMEOUT_SECONDS = float(os.environ.get("OLLAMA_TIMEOUT_SECONDS", "180"))
OPENROUTER_TIMEOUT_SECONDS = float(os.environ.get("OPENROUTER_TIMEOUT_SECONDS", "120"))
TEST_QUESTIONS_LIMIT = int(os.environ.get("TEST_QUESTIONS_LIMIT", "0"))
OPENROUTER_HEADERS = {
    "HTTP-Referer": "http://localhost:3000",
    "X-Title": "Sql Oil Assistant",
}


class AgentState(TypedDict):
    question: str
    generated_sql: str
    error_message: str
    retry_count: int
    query_result: str
    query_column_context: str
    sql_generation_time: float
    sql_execution_time: float
    remote_response_time: float
    local_prompt_chars: int
    remote_prompt_chars: int
    final_response: str



def safe_str(text: str | None) -> str:
    if text is None:
        return ""
    return str(text).encode("utf-8", errors="ignore").decode("utf-8")



def estimate_message_chars(messages: List[Dict[str, str]]) -> int:
    total_chars = 0
    for message in messages:
        total_chars += len(safe_str(message.get("role", "")))
        total_chars += len(safe_str(message.get("content", "")))
    return total_chars



def log_progress(message: str) -> None:
    print(safe_str(message), flush=True)



def format_llm_error(exc: Exception) -> str:
    parts = [f"{type(exc).__name__}: {safe_str(str(exc))}"]

    status_code = getattr(exc, "status_code", None)
    if status_code is not None:
        parts.append(f"status={status_code}")

    response = getattr(exc, "response", None)
    if response is not None:
        response_text = safe_str(getattr(response, "text", ""))[:400]
        if response_text:
            parts.append(f"body={response_text}")

    return " | ".join(part for part in parts if part)



def format_local_error(exc: Exception) -> str:
    if isinstance(exc, ollama.RequestError):
        return (
            f"Ollama indisponível em {OLLAMA_HOST}. "
            f"Inicie com `ollama serve`. Detalhe: {safe_str(str(exc))}"
        )

    if isinstance(exc, ollama.ResponseError):
        if getattr(exc, "status_code", None) == 404:
            return (
                f"Modelo local '{LOCAL_SQL_MODEL}' não encontrado. "
                f"Baixe com `ollama pull {LOCAL_SQL_MODEL}`."
            )
        return f"ResponseError(status={exc.status_code}): {safe_str(str(exc))}"

    return f"{type(exc).__name__}: {safe_str(str(exc))}"



def extract_text_content(content: Any) -> str:
    if isinstance(content, str):
        return safe_str(content)

    if isinstance(content, list):
        parts: List[str] = []
        for item in content:
            if isinstance(item, str):
                parts.append(safe_str(item))
                continue

            if isinstance(item, dict) and item.get("type") == "text":
                text = item.get("text", "")
                if text:
                    parts.append(safe_str(text))

        return "\n".join(part for part in parts if part).strip()

    return safe_str(content)


if not OPENROUTER_API_KEY:
    print("[Aviso] OPENROUTER_API_KEY não encontrada no ambiente.")

ollama_client = ollama.Client(host=OLLAMA_HOST, timeout=OLLAMA_TIMEOUT_SECONDS)
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY or "missing_api_key",
    base_url=OPENROUTER_BASE_URL,
    timeout=OPENROUTER_TIMEOUT_SECONDS,
)

LOCAL_SQL_MODEL_READY = False
LOCAL_SQL_SYSTEM_PROMPT = (
    "Você é um engenheiro de software especialista em banco de dados, "
    "com especialidade em SQLite. "
    "Sua função neste fluxo é traduzir perguntas em linguagem natural "
    "para consultas SQL corretas, seguras e compatíveis com SQLite. "
    "Use apenas o esquema fornecido e retorne somente SQL puro quando solicitado. "
    "Jamais retorne nenhum tipo de formatação, como por exemplo markdown. "
    "Padrões obrigatórios de execução: use apenas SELECT; prefira ORDER BY ... DESC/ASC LIMIT 1 para perguntas de máximo ou mínimo quando a resposta exigir a data associada; "
    "evite SELECT com MAX/MIN junto de colunas não agregadas sem ORDER BY ou subquery explícita; sempre use aliases legíveis para agregações; "
    "se a pergunta pedir média, soma, máximo, mínimo ou acumulado, nomeie a coluna de saída de forma clara; "
    "se a pergunta for temporal, preserve DATEPRD no resultado; "
    "nunca invente colunas, unidades, entidades operacionais ou lógica fora do esquema e do dicionário. "
)



def probe_local_sql_runtime() -> str:
    try:
        ollama_client.show(LOCAL_SQL_MODEL)
        return f"[LOCAL SQL] Modelo disponível no Ollama: {LOCAL_SQL_MODEL}"
    except Exception as exc:
        return f"[LOCAL SQL] {format_local_error(exc)}"



def ensure_local_sql_model_ready() -> None:
    global LOCAL_SQL_MODEL_READY
    if LOCAL_SQL_MODEL_READY:
        return

    try:
        ollama_client.show(LOCAL_SQL_MODEL)
        LOCAL_SQL_MODEL_READY = True
    except Exception as exc:
        raise RuntimeError(format_local_error(exc)) from exc



def invoke_local_sql_model(prompt: str) -> str:
    ensure_local_sql_model_ready()

    response = ollama_client.chat(
        model=LOCAL_SQL_MODEL,
        messages=[
            {"role": "system", "content": LOCAL_SQL_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        options={
            "temperature": 0,
            "num_predict": 220,
            "num_ctx": 8192,
        },
        keep_alive="30m",
    )

    content = extract_text_content(response.message.content)
    if not content:
        raise ValueError("O modelo local retornou conteúdo vazio.")

    return content



def invoke_openrouter(prompt: str) -> str:
    if not OPENROUTER_API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY não encontrada no ambiente.")

    response = openrouter_client.chat.completions.create(
        model=REMOTE_RESPONSE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        extra_headers=OPENROUTER_HEADERS,
    )

    if not response.choices:
        raise ValueError("OpenRouter retornou uma resposta sem choices.")

    message = response.choices[0].message
    content = extract_text_content(message.content)
    if not content:
        raise ValueError("OpenRouter retornou conteúdo vazio.")

    return content


print(probe_local_sql_runtime())
print(f"[REMOTE RESPONSE] Modelo configurado: {REMOTE_RESPONSE_MODEL}")

[LOCAL SQL] Modelo disponível no Ollama: qwen2.5-coder:7b-instruct
[REMOTE RESPONSE] Modelo configurado: anthropic/claude-sonnet-4.6


### Célula 3: Nós do fluxo e compilação do LangGraph

In [3]:
from textwrap import dedent

PROHIBITED_KEYWORDS = {
    "DROP",
    "DELETE",
    "INSERT",
    "UPDATE",
    "ALTER",
    "CREATE",
    "TRUNCATE",
    "EXECUTE"
}



def generate_sql_node(state: AgentState) -> Dict[str, Any]:
    error_message = state.get("error_message", "")
    error_context = ""
    if error_message:
        error_context = (
            "\nATENÇÃO: sua tentativa anterior falhou com o erro: "
            f"{error_message}. Corrija a sintaxe."
        )

    prompt = dedent(
        f"""
        Sua tarefa é gerar SQL SQLite para uma série temporal real de produção offshore do projeto Volve, no Mar do Norte da Noruega.

        Tabela disponível: {TABLE_NAME}

        Esquema:
        {schema_text}

        Dicionário de dados:
        {DATA_DICTIONARY}

        Regras:
        1. Retorne somente SQL puro, sem markdown.
        2. Use apenas SELECT.
        3. Use exatamente os nomes das colunas disponíveis no esquema.
        4. Use LIMIT em vez de TOP.
        5. Quando a pergunta pedir máximo, mínimo, média, tendência ou ranking temporal, retorne também as colunas de apoio necessárias para interpretar o resultado.
        6. Se a pergunta mencionar data ou depender de contexto temporal, inclua a coluna DATEPRD no resultado quando isso for necessário.
        7. Para máximo/mínimo com data associada, prefira ORDER BY com LIMIT 1 em vez de MAX/MIN com coluna não agregada solta.
        8. Dê aliases claros para colunas derivadas, por exemplo avg_on_stream_hrs, max_water_cumulative, max_oil_roll_30.
        9. Nunca invente colunas, tabelas, datas, unidades ou métricas que não existam no esquema.
        {error_context}

        Pergunta: {state["question"]}
        SQL:
        """
    ).strip()
    local_messages = [
        {"role": "system", "content": LOCAL_SQL_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    local_prompt_chars = estimate_message_chars(local_messages)
    attempt_number = state.get("retry_count", 0) + 1
    log_progress(
        f"[SQL][INICIO] tentativa={attempt_number} pergunta={safe_str(state['question'])}"
    )
    sql_generation_start_time = time.time()

    try:
        clean_sql = (
            invoke_local_sql_model(safe_str(prompt))
            .replace("```sql", "")
            .replace("```", "")
            .replace(";", "")
            .strip()
        )
        sql_generation_elapsed = time.time() - sql_generation_start_time
        log_progress(
            f"[SQL][OK] tentativa={attempt_number} tempo={sql_generation_elapsed:.2f}s"
        )
        return {
            "generated_sql": safe_str(clean_sql),
            "retry_count": state.get("retry_count", 0) + 1,
            "sql_generation_time": state.get("sql_generation_time", 0.0) + sql_generation_elapsed,
            "local_prompt_chars": state.get("local_prompt_chars", 0) + local_prompt_chars,
        }
    except Exception as exc:
        sql_generation_elapsed = time.time() - sql_generation_start_time
        log_progress(
            f"[SQL][ERRO] tentativa={attempt_number} tempo={sql_generation_elapsed:.2f}s detalhe={safe_str(str(exc))}"
        )
        return {
            "error_message": f"Erro no modelo local de SQL: {safe_str(str(exc))}",
            "retry_count": state.get("retry_count", 0) + 1,
            "sql_generation_time": state.get("sql_generation_time", 0.0) + sql_generation_elapsed,
            "local_prompt_chars": state.get("local_prompt_chars", 0) + local_prompt_chars,
        }



def execute_sql_node(state: AgentState) -> Dict[str, Any]:
    error_message = state.get("error_message", "")
    if "Erro no modelo local de SQL" in error_message:
        return {"query_result": ""}

    generated_sql = state.get("generated_sql", "").strip()
    if not generated_sql:
        return {
            "error_message": "Nenhum SQL válido foi gerado.",
            "query_result": "",
            "query_column_context": "",
        }

    sql_to_run = generated_sql.upper()
    if any(keyword in sql_to_run for keyword in PROHIBITED_KEYWORDS):
        return {
            "error_message": "Bloqueio de segurança: comando de escrita proibido.",
            "query_result": "",
            "query_column_context": "",
        }

    sql_execution_start_time = time.time()
    log_progress(f"[SQLITE][INICIO] executando SQL para pergunta={safe_str(state['question'])}")
    try:
        df = pd.read_sql(generated_sql, conn)
        query_columns = [safe_str(str(column)) for column in df.columns.tolist()]
        sql_execution_elapsed = time.time() - sql_execution_start_time
        log_progress(
            f"[SQLITE][OK] linhas={len(df)} colunas={len(query_columns)} tempo={sql_execution_elapsed:.2f}s"
        )
        return {
            "query_result": safe_str(df.to_string(index=False)),
            "query_column_context": safe_str(build_query_column_context(query_columns)),
            "error_message": "",
            "sql_execution_time": state.get("sql_execution_time", 0.0) + sql_execution_elapsed,
        }
    except Exception as exc:
        sql_execution_elapsed = time.time() - sql_execution_start_time
        log_progress(
            f"[SQLITE][ERRO] tempo={sql_execution_elapsed:.2f}s detalhe={safe_str(str(exc))}"
        )
        return {
            "error_message": safe_str(str(exc)),
            "query_result": "",
            "query_column_context": "",
            "sql_execution_time": state.get("sql_execution_time", 0.0) + sql_execution_elapsed,
        }



def respond_node(state: AgentState) -> Dict[str, Any]:
    error_message = state.get("error_message", "")
    if error_message:
        return {
            "final_response": (
                "Não foi possível responder devido a um erro persistente: "
                f"{error_message}"
            )
        }

    prompt = dedent(
        f"""
        Você é um engenheiro sênior de produção de petróleo e gás natural em ambiente offshore.

        Responda usando somente os dados retornados abaixo.

        Regras:
        1. Não invente datas, comparações, interpretações operacionais ou explicações não presentes nos dados.
        2. Se a consulta retornou apenas o primeiro registro, não extrapole além do que o próprio resultado mostra.
        3. Preserve os nomes das colunas exatamente como vieram do resultado SQL.
        4. Use o dicionário de dados e os metadados das colunas retornadas para interpretar conceitos de produção de petróleo, gás, água e séries temporais.
        5. Quando a unidade estiver disponível no dicionário, use a unidade correta explicitamente na resposta.
        6. Não invente ou converta unidades além do que o dicionário informa.
        7. Cite os valores exatos retornados.
        8. Seja curto, técnico e conclusivo.

        Pergunta: {state["question"]}
        Dicionário de dados:
        {DATA_DICTIONARY}

        Metadados das colunas retornadas:
        {state.get("query_column_context", "")}

        Dados do banco:
        {state["query_result"]}
        Resposta técnico-comercial:
        """
    ).strip()
    remote_messages = [{"role": "user", "content": prompt}]
    remote_prompt_chars = estimate_message_chars(remote_messages)
    log_progress(f"[REMOTE][INICIO] pergunta={safe_str(state['question'])}")
    remote_response_start_time = time.time()

    try:
        final_response = invoke_openrouter(safe_str(prompt))
        remote_response_elapsed = time.time() - remote_response_start_time
        log_progress(f"[REMOTE][OK] tempo={remote_response_elapsed:.2f}s")
        return {
            "final_response": safe_str(final_response.strip()),
            "remote_response_time": state.get("remote_response_time", 0.0) + remote_response_elapsed,
            "remote_prompt_chars": state.get("remote_prompt_chars", 0) + remote_prompt_chars,
        }
    except Exception as exc:
        remote_response_elapsed = time.time() - remote_response_start_time
        log_progress(
            f"[REMOTE][ERRO] tempo={remote_response_elapsed:.2f}s detalhe={safe_str(str(exc))}"
        )
        return {
            "final_response": (
                "Erro na geração da resposta final: "
                f"{format_llm_error(exc)}"
            ),
            "remote_response_time": state.get("remote_response_time", 0.0) + remote_response_elapsed,
            "remote_prompt_chars": state.get("remote_prompt_chars", 0) + remote_prompt_chars,
        }



def should_retry_or_respond(state: AgentState) -> str:
    if state.get("error_message") and state.get("retry_count", 0) < 3:
        return "generate_sql"
    return "respond"


workflow = StateGraph(AgentState)
workflow.add_node("generate_sql", generate_sql_node)
workflow.add_node("execute_sql", execute_sql_node)
workflow.add_node("respond", respond_node)

workflow.add_edge(START, "generate_sql")
workflow.add_edge("generate_sql", "execute_sql")
workflow.add_conditional_edges(
    "execute_sql",
    should_retry_or_respond,
    {"generate_sql": "generate_sql", "respond": "respond"},
)
workflow.add_edge("respond", END)

app = workflow.compile()
print("[DIAGNÓSTICO] Grafo híbrido compilado: SQL local + resposta final remota.")

[DIAGNÓSTICO] Grafo híbrido compilado: SQL local + resposta final remota.


### Célula 4: Teste de estresse e geração do relatório

In [4]:
test_questions = [
    "Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor?",
    "Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor?",
    "Qual foi a média de ON_STREAM_HRS?",
    "Em qual data ocorreu a maior AVG_DOWNHOLE_PRESSURE e qual foi o valor?",
    "Qual foi o maior valor de water_cumulative e em qual data ocorreu?",
    "Qual foi o maior valor de oil_roll_30 e em qual data ocorreu?",
]

if TEST_QUESTIONS_LIMIT > 0:
    test_questions = test_questions[:TEST_QUESTIONS_LIMIT]

report_dir = os.path.join(os.getcwd(), "notebooks")
if os.path.isdir(report_dir):
    REPORT_PATH = os.path.join(report_dir, "relatorio_dados_reais_producao_volve_noruega.md")
else:
    REPORT_PATH = os.path.abspath("relatorio_dados_reais_producao_volve_noruega.md")

results: List[Dict[str, Any]] = []
total_start_time = time.time()

print("=" * 80, flush=True)
print(f"INICIANDO TESTE DE ESTRESSE COM {len(test_questions)} PERGUNTAS", flush=True)
print("=" * 80, flush=True)

for index, question in enumerate(test_questions, start=1):
    print(f"[TESTE {index}/{len(test_questions)}] {question}", flush=True)

    initial_state: AgentState = {
        "question": safe_str(question),
        "generated_sql": "",
        "error_message": "",
        "retry_count": 0,
        "query_result": "",
        "query_column_context": "",
        "sql_generation_time": 0.0,
        "sql_execution_time": 0.0,
        "remote_response_time": 0.0,
        "local_prompt_chars": 0,
        "remote_prompt_chars": 0,
        "final_response": "",
    }

    step_start_time = time.time()
    output = app.invoke(initial_state, {"recursion_limit": 15})
    elapsed = time.time() - step_start_time
    print(f"[TESTE {index}/{len(test_questions)}][FIM] tempo_total={elapsed:.2f}s", flush=True)

    results.append(
        {
            "index": index,
            "question": question,
            "elapsed": elapsed,
            "total_time": elapsed,
            "retry_count": max(output.get("retry_count", 0) - 1, 0),
            "generated_sql": output.get("generated_sql", ""),
            "query_result": output.get("query_result", ""),
            "error_message": output.get("error_message", ""),
            "sql_generation_time": output.get("sql_generation_time", 0.0),
            "sql_execution_time": output.get("sql_execution_time", 0.0),
            "remote_response_time": output.get("remote_response_time", 0.0),
            "local_prompt_chars": output.get("local_prompt_chars", 0),
            "remote_prompt_chars": output.get("remote_prompt_chars", 0),
            "final_response": output.get("final_response", ""),
        }
    )

total_duration = time.time() - total_start_time
success_count = sum(1 for result in results if not result["error_message"])
failed_count = len(results) - success_count
overall_status = (
    "Concluído com sucesso"
    if failed_count == 0
    else f"Concluído com falhas ({failed_count}/{len(results)})"
)

with open(REPORT_PATH, "w", encoding="utf-8") as md:
    md.write("# Relatório Executivo - Dados Reais de Produção Volve (Noruega)\n\n")
    md.write(f"**Data da Execução:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    md.write(f"**Fonte de Dados CSV:** {CSV_FILE_NAME}\n")
    md.write(f"**Banco SQLite:** {DB_NAME}\n")
    md.write(f"**Tabela SQLite:** {TABLE_NAME}\n")
    md.write(f"**Shape do DataFrame:** {source_df.shape}\n\n")
    md.write(f"**Modelo Local (SQL):** {LOCAL_SQL_MODEL} via Ollama\n")
    md.write(f"**Modelo Remoto (Resposta Final):** {REMOTE_RESPONSE_MODEL} via OpenRouter\n\n")
    md.write("## 1. Fonte dos Dados\n\n")
    md.write("```text\n")
    md.write(DATA_SOURCE_TEXT)
    md.write("\n```\n\n")
    md.write("## 2. Dicionário de Dados Utilizado\n\n")
    md.write("```text\n")
    md.write(DATA_DICTIONARY)
    md.write("\n```\n\n")
    md.write("## 3. Histórico de Execuções e Respostas Técnicas\n\n")

    for result in results:
        case_index = result["index"]
        question_text = safe_str(result["question"])
        total_time_text = "{:.2f}".format(result["total_time"])
        sql_generation_time_text = "{:.2f}".format(result["sql_generation_time"])
        sql_execution_time_text = "{:.2f}".format(result["sql_execution_time"])
        remote_response_time_text = "{:.2f}".format(result["remote_response_time"])
        local_prompt_chars_text = str(result["local_prompt_chars"])
        remote_prompt_chars_text = str(result["remote_prompt_chars"])
        retry_count = result["retry_count"]
        status_text = safe_str(result["error_message"]) or "Sem erros"
        sql_text = safe_str(result["generated_sql"])
        query_result_text = safe_str(result["query_result"])
        final_response_text = safe_str(result["final_response"])

        md.write(f"### Caso de Teste {case_index}: {question_text}\n")
        md.write(f"- Tempo Total: {total_time_text} segundos\n")
        md.write(f"- Tempo de Geração do SQL: {sql_generation_time_text} segundos\n")
        md.write(f"- Tempo de Execução SQL: {sql_execution_time_text} segundos\n")
        md.write(f"- Tempo de Resposta Remota: {remote_response_time_text} segundos\n")
        md.write(f"- Tamanho do Prompt Local Enviado: {local_prompt_chars_text} caracteres\n")
        md.write(f"- Tamanho do Prompt Remoto Enviado: {remote_prompt_chars_text} caracteres\n")
        md.write(f"- Tentativas de Correção (Retries): {retry_count}\n")
        md.write(f"- Status: {status_text}\n\n")
        md.write("```sql\n")
        md.write(f"{sql_text}\n")
        md.write("```\n\n")
        md.write("```text\n")
        md.write(f"{query_result_text}\n")
        md.write("```\n\n")
        md.write(f"> {final_response_text}\n\n")
        md.write("---\n\n")

    md.write("## 4. Sumário Executivo de Performance\n\n")
    md.write(f"- Total de Perguntas Submetidas: {len(results)}\n")
    md.write(f"- Casos com sucesso: {success_count}\n")
    md.write(f"- Casos com falha: {failed_count}\n")
    md.write(f"- Tempo Total de Varredura: {total_duration:.2f} segundos\n")
    md.write(f"- Média de Tempo por Requisição: {total_duration / len(results):.2f} segundos\n")
    md.write(f"- Média de Tempo de Geração do SQL: {sum(result['sql_generation_time'] for result in results) / len(results):.2f} segundos\n")
    md.write(f"- Média de Tempo de Execução SQL: {sum(result['sql_execution_time'] for result in results) / len(results):.2f} segundos\n")
    md.write(f"- Média de Tempo de Resposta Remota: {sum(result['remote_response_time'] for result in results) / len(results):.2f} segundos\n")
    md.write(f"- Média de Tamanho do Prompt Local: {sum(result['local_prompt_chars'] for result in results) / len(results):.2f} caracteres\n")
    md.write(f"- Média de Tamanho do Prompt Remoto: {sum(result['remote_prompt_chars'] for result in results) / len(results):.2f} caracteres\n")
    md.write(f"- Status Geral do Sistema: {overall_status}\n")

print("=" * 80, flush=True)
print(overall_status, flush=True)
print(f"Relatório salvo em: {REPORT_PATH}", flush=True)
print("=" * 80, flush=True)

INICIANDO TESTE DE ESTRESSE COM 6 PERGUNTAS
[TESTE 1/6] Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor?
[SQL][INICIO] tentativa=1 pergunta=Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor?
[SQL][OK] tentativa=1 tempo=47.68s
[SQLITE][INICIO] executando SQL para pergunta=Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor?
[SQLITE][OK] linhas=1 colunas=2 tempo=0.00s
[REMOTE][INICIO] pergunta=Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor?
[REMOTE][OK] tempo=6.95s
[TESTE 1/6][FIM] tempo_total=54.65s
[TESTE 2/6] Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor?
[SQL][INICIO] tentativa=1 pergunta=Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor?
[SQL][OK] tentativa=1 tempo=39.60s
[SQLITE][INICIO] executando SQL para pergunta=Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor?
[SQLITE][OK] linhas=1 colunas=2 tempo=0.00s
[REMOTE][INICIO] pergunta=Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor?
[RE